# Qlib 数据与因子探索

**第一部分 · 课堂讲解版**

本 Notebook 只负责环境、Qlib 初始化、A 股数据查询、表达式特征以及一个月的 Alpha158 小样本探索。它不会创建正式训练集，也不会训练模型。完成后请断开并删除运行时，再打开第二部分，让训练与回测从干净内存开始。

## 1. 云端环境准备

Colab 会为每位学习者提供临时 Python 环境，因此不需要在本地安装 Qlib。下面的初始化单元格会：

- 固定使用 Colab 2026.07 运行时（Python 3.12），避免 Qlib 与 Python 3.13 不兼容；
- 安装与本教程验证版本一致的 Qlib、Statsmodels 和 LightGBM，并把 Plotly 放在 `/content` 隔离目录中；
- 从 Qlib README 当前推荐的社区镜像下载 A 股教学数据；
- 将数据解压到 Colab 的 `/content/qlib_data/cn_data`；
- 检查 Python 版本和数据目录是否就绪。

> Colab 虚拟机是临时的。运行时被回收后，依赖与数据需要重新准备。
> 本教程采用低内存流程。不要把示例中的小样本改为三份全量数据同时常驻内存。

**连接故障快速判断**

如果在执行任何单元格之前就出现 `/api/kernelspecs` 500，请先新建一个空白 Colab 并运行 `print("ok")`。空白 Notebook 也失败，说明是 Colab 会话、账号配额或网络连接问题，不是本教程代码；请删除当前运行时、重新连接 CPU 运行时后再试。错误链接包含临时运行时令牌，不要公开转发。

In [ ]:
#@title 运行一次：准备 Qlib 教学环境
import sys
import subprocess
from pathlib import Path
from IPython.display import display

if sys.version_info[:2] > (3, 12):
    raise RuntimeError(
        "请将 Colab 运行时版本切换为 2026.07（Python 3.12）后重新运行。"
    )

QLIB_DATA_DIR = Path("/content/qlib_data/cn_data")
PLOTLY_DIR = Path("/content/qlib_plotly_runtime")

print("[1/3] 安装 Python 依赖……")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "pyqlib==0.9.7", "statsmodels==0.14.6", "lightgbm==4.6.0",
])

# Plotly 单独安装到 /content，避开 Colab 中可能损坏的系统包。
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "--upgrade", "--target", str(PLOTLY_DIR), "plotly==6.6.0",
])
sys.path.insert(0, str(PLOTLY_DIR))

# 清除失败导入留下的缓存，再从隔离目录加载 Plotly。
for module_name in list(sys.modules):
    if module_name == "plotly" or module_name.startswith(("plotly.", "_plotly_utils")):
        sys.modules.pop(module_name, None)

calendar_file = QLIB_DATA_DIR / "calendars" / "day.txt"
print("[2/3] 检查 Qlib A 股数据……")
if not calendar_file.exists():
    archive = Path("/content/qlib_bin.tar.gz")
    QLIB_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("下载教学数据（约 464 MB）……")
    subprocess.check_call([
        "wget", "-q", "--show-progress",
        "https://github.com/chenditc/investment_data/releases/latest/download/qlib_bin.tar.gz",
        "-O", str(archive),
    ])
    subprocess.check_call(
        ["tar", "-xzf", str(archive), "-C", str(QLIB_DATA_DIR), "--strip-components=1"],
    )
    archive.unlink(missing_ok=True)

if not calendar_file.exists():
    raise FileNotFoundError(f"Qlib 数据准备失败：{QLIB_DATA_DIR}")

print("[3/3] 配置 Plotly……")
import plotly
import plotly.io as pio
pio.renderers.default = "colab"

def show_plotly(figure):
    figure.show(renderer="colab", config={"responsive": True})

print(f"Python: {sys.version.split()[0]}")
print(f"Plotly: {plotly.__version__}")
print(f"Qlib 数据目录: {QLIB_DATA_DIR}")
print("环境准备完成 ✓")

In [ ]:
import sys, site
from pathlib import Path

In [ ]:
import qlib
import pandas as pd
from qlib.constant import REG_CN

## 2. 框架初始化

在这一部分，我们将初始化 Qlib 框架，设置数据路径和股票池。这是整个工作流的第一步。

**学习目标**：
- 理解如何初始化 Qlib 框架
- 理解股票池（market）和基准指数（benchmark）

**步骤**1: 设置数据存储路径 

In [ ]:
# provider_uri: 指定 qlib 数据的存储位置（与数据下载的target_dir一致）
provider_uri = str(QLIB_DATA_DIR)  # Colab 与本地共用

**步骤**2: 初始化 Qlib 系统

**概念说明**：
- `qlib.init()` 是 Qlib 的核心初始化函数，用于启动整个框架
- 需要指定数据存储路径（provider_uri）和市场区域（region） 
    - region：不同模式会导致不同的交易限制和成本。region 只是用于定义一批配置的快捷方式，包括最小交易单位（trade_unit）、交易限制（limit_threshold）等。它不是必需的，如果现有的 region 设置无法满足需求，用户可以手动设置关键配置。
- 初始化后，才能使用 Qlib 的各种功能

In [ ]:
# ==================== 初始化 Qlib 框架 ====================
# 启动 Qlib 框架，指定数据路径和市场区域

# 参数说明：
# provider_uri：数据存储路径，Qlib 会从这个路径读取股票数据
# region：市场区域，REG_CN 表示中国市场（A股）

# 本教程仅面向 Colab，直接把 MLflow 数据库放在临时运行时目录。
mlflow_db = Path("/content/qlib_mlflow.db")
exp_manager = {
    "class": "MLflowExpManager",
    "module_path": "qlib.workflow.expm",
    "kwargs": {
        "uri": f"sqlite:///{mlflow_db.resolve().as_posix()}",
        "default_exp_name": "Experiment",
    },
}

qlib.init(provider_uri=provider_uri, region=REG_CN, exp_manager=exp_manager, kernels=1)
print(f"MLflow 实验数据库: {mlflow_db}")

**步骤3**: 指定股票池

In [ ]:
# ==================== 股票池和基准指数配置 ====================

market = "csi300"        # 股票池（指定要研究的股票集合）：沪深300成分股
benchmark = "SH000300"   # 基准指数（指定要对比的指数）：沪深300指数

## 3. 数据探索

在这一部分，我们将学习如何探索和了解 Qlib 中的股票数据。在开始正式的数据处理之前，先了解数据的结构对理解后续步骤很重要。

**学习目标**：
- 理解如何查看股票列表和特征数据
- 了解原始特征和派生特征的区别
- 理解动态成分股的概念（股票池会随时间变化）

In [ ]:
# D对象（"Data"缩写）是Qlib中的全局实例，为数据检索提供统一接口。调用qlib.init()后，D即可响应数据请求。
from qlib.data import D
from pprint import pprint

### 3.1 查看成分股列表

对于大部分指数来说，其成分股列表都不是一成不变的。因此，我们需要指定一个时间段，然后查看成分股列表在该时间段包含哪些公司。

In [ ]:
D.instruments('csi300')

In [ ]:
# 查询一年沪深指数成分股
instruments_dict = D.list_instruments(
    instruments=D.instruments('csi300'),
    start_time="2019-01-01",
    end_time="2019-12-31"
)
print('总数量', len(instruments_dict))

In [ ]:
# 使用 DataFrame 展示部分成分股及其生效区间
preview_items = list(instruments_dict.items())
rows = []
for inst, spans in preview_items:
    for span in spans:
        start_date = span[0]
        end_date = span[1]
        rows.append({
            'instrument': inst,
            'start_date': start_date,
            'end_date': end_date,
        })
preview_df = pd.DataFrame(rows)
display(preview_df)

In [ ]:
instruments_dict = D.list_instruments(
    instruments=D.instruments("csi300"),
    start_time="2019-02-04",
    end_time="2019-02-04",
    as_list=True # as_list=True：返回列表（list）；默认值as_list=False：返回字典（dict），键是股票代码，值是该股票在指定时间段内的“生效区间”
)
print(len(instruments_dict))

### 3.2 查看特征

#### 原始特征

In [ ]:
# ============================================
# 查看预置数据的全部原始特征
# ============================================

# 查看某个股票的可用特征
stock_features_path = QLIB_DATA_DIR / "features" / "sh600000"  # 以浦发银行为例

# 列出所有特征文件
feature_files = list(stock_features_path.glob("*.day.bin"))
features = [f.stem.split('.')[0] for f in feature_files]
print("可用的原始特征:")
for feature in sorted(features):
    print(f"  ${feature}", end=" ")

总结：qlib 中国市场数据（通过 `qlib-data` 下载的标准数据），**通常包含以下原始特征**：

In [ ]:
fields_basic = [
    "$open",      # 开盘价
    "$high",      # 最高价
    "$low",       # 最低价
    "$close",     # 收盘价
    "$volume",    # 成交量
    "$factor",    # 复权因子
    "$change",    # 涨跌额
]

**注意事项：**
1. 不同数据源可能提供不同的字段
2. 高频数据（如分钟线）可能有额外字段
3. 自定义数据可以包含任意字段

In [ ]:
# ============================================
# 示例1: 查看单个股票的单个特征
# ============================================
df1 = D.features(
    instruments=["SH600000"],  # 浦发银行
    fields=["$close"],         # 收盘价
    start_time="2020-01-01",
    end_time="2020-12-31"
)
display(df1.head(5))

In [ ]:
# ============================================
# 示例2: 查看多个股票的多个特征
# ============================================
df2 = D.features(
    instruments=["SH600000", "SH600016", "SH600019"],
    fields=fields_basic,
    start_time="2020-01-01",
    end_time="2020-01-03"
)
display(df2)

In [ ]:
# ============================================
# 示例3: 使用市场指数查看所有成分股
# ============================================
df3 = D.features(
    instruments=D.instruments("csi300"),  # CSI300所有成分股
    fields=["$close"],
    start_time="2020-01-01",
    end_time="2020-01-03"
)
print(f"数据形状: {df3.shape}")
print(f"涉及股票数: {len(df3.index.get_level_values('instrument').unique())}")
display(df3.head(10))

In [ ]:
# ============================================
# 示例4: 不指定时间范围（查看所有可用数据）
# ============================================
df4 = D.features(
    instruments=["SH600000"],
    fields=["$close"]
    # 不指定 start_time 和 end_time
)
print(f"数据起始: {df4.index.get_level_values('datetime').min()}")
print(f"数据结束: {df4.index.get_level_values('datetime').max()}")
print(f"总记录数: {len(df4)}")

#### 派生特征

**概念说明**：
- **派生特征**：基于原始特征计算得到的衍生指标
- 例如：移动平均线（MA）、相对强弱指标（RSI）等
- 这些特征通过数学运算从原始特征中提取，通常包含更多信息

In [ ]:
# ============================================
# 示例5: 使用表达式计算派生特征
# ============================================
df5 = D.features(
    instruments=["SH600000", "SH600016"],
    fields=[
        "$close",                                    # 原始收盘价
        "Ref($close, 1)",                            # 前一天收盘价
        "($close - Ref($close, 1)) / Ref($close, 1)",  # 日收益率
        "Mean($close, 5)",                           # 5日均价
        "Std($close, 20)",                           # 20日标准差
        # MACD 相关
        "EMA($close, 12)",                           # 12日指数移动平均
        "EMA($close, 26)",                           # 26日指数移动平均
        "EMA($close, 12) - EMA($close, 26)",         # MACD DIF
        # 布林带
        "Mean($close, 20) + 2 * Std($close, 20)",    # 上轨
        "Mean($close, 20) - 2 * Std($close, 20)",    # 下轨
        # 动量指标
        "$close / Ref($close, 10) - 1",              # 10日动量
    ],
    start_time="2020-01-01",
    end_time="2020-01-31"
    )
# 可以重命名列
df5.columns = ['close', 'close_lag1', 'return', 'ma5', 'std20', 'ema12', 'ema26', 'macd_dif', 
               'bb_upper', 'bb_lower', 'momentum10']
display(df5.head(5))

注意到`D.features`返回值的类型是pandas dataframe，因此dataframe的相关处理技巧也适用于`D.features`的返回值。

In [ ]:
# ============================================
# 示例6: 实际应用 - 计算相关性矩阵
# ============================================
df6 = D.features(
    instruments=D.instruments("csi300"),
    fields=["$close"],
    start_time="2019-01-01",
    end_time="2019-12-31"
)

# 转换为宽格式并计算收益率
df6 = df6.unstack(level='instrument')
display(df6.head())

returns = df6.pct_change(fill_method=None)
correlation_matrix = returns.corr()

print(f"相关性矩阵形状: {correlation_matrix.shape}")
display(correlation_matrix.iloc[:5, :5])  # 显示前5×5

更多可选运算符，可以阅读`qlib.data.ops`的帮助文档。

### **课后作业（1）：股票筛选**
参考链接: https://qlib.readthedocs.io/en/latest/reference/api.html#data

股票池：csi300
时间：2015-01-01 到 2016-02-15

1. 使用 NameDFilter 从中筛选出只在上交所（SH）上市的成分股（即代码以 SH 开头），计算占比

In [ ]:
# 请补充代码

2. 使用 ExpressionDFilter，筛选收盘价曾大于 1000 的股票

In [ ]:
# 请补充代码

3. 使用 ExpressionDFilter，筛选过去 60 日涨幅超过 50% 的股票

In [ ]:
# 请补充代码

## 4. 数据准备

在这一部分，我们将准备用于模型训练的数据。这是整个工作流的核心步骤。

**学习目标**
- 理解如何配置数据处理器（DataHandler）
- 理解如何配置数据集（DatasetH）
- 了解数据分割和归一化的基本概念

**DataHandler（数据处理器）**
- 负责从原始数据中提取特征和标签，并进行数据标准化

**DatasetH（数据集）**
- 基于 DataHandler 创建，负责数据分割（train/valid/test）和数据获取

**示例任务**
- 构造基于Alpha158特征集的数据集用于跨日收益预测模型训练

*Alpha158特征集*
- Qlib量化投资平台提供的一个经典特征集，包含158个经过精心设计的量化因子。这些因子主要基于股票的开盘价、最高价、最低价、收盘价和成交量等基础数据，通过人工特征工程方法提取而成
    - **KBAR特征**（9个）：K线形态特征，如 `($close-$open)/$open` 等
    - **PRICE特征**（4个）：原始价格特征，如 `$open/$close`, `$high/$close` 等
    - **ROLLING特征**（145个）：滚动统计特征，包含约30种技术指标
        - 使用 `windows=[5, 10, 20, 30, 60]` 天的5种滚动窗口计算
        - 包括趋势类指标（ROC, MA, BETA等）、波动性指标（STD, RSQR等）、极值指标（MAX, MIN等）、动量指标以及量价关系（CORR, CORD等）

*跨日收益率预测任务*
- 使用 T 日及之前的数据（特征）预测 T+1 → T+2 的收益率（标签）
    - 预测本身使用了 T 日的收盘价信息
    - 当天收盘后才能进行预测
    - 对应的策略需要在 T 日收盘后进行预测及分析，然后在 T+1 日开盘后进行交易，更符合实际交易场景
- 指标计算时间线图解

``` text
时间轴:
  T-2      T-1       T       T+1            T+2
   │        │        │       |              │
   │        │      今天       │              │
   │        │     (特征)      │              │
   │        │        │       │              │ 
   │        │        │     Close₁         Close₂
   │        │        │        ↓            ↓
   │        │        │    Ref($close,-1)  Ref($close,-2)
   │        │        │                ↓
   │        │        └────────────> 标签 = Close₂/Close₁ - 1
```


*数据集构造流程*
- 原始特征及标签提取 -> 数据标准化 -> 数据集划分

### 4.1 配置数据处理器（DataHandler）

数据处理器负责从原始数据中提取**特征**和**标签**，并进行**数据标准化**。我们使用 Qlib 提供的 `Alpha158` 数据处理器，它会自动构建 158 个特征以及对应跨日收益率标签

**步骤 1：配置参数**

首先，我们需要配置数据加载的参数。

In [ ]:
# ==================== 小样本探索配置 ====================
def show_process_memory(stage):
    """显示当前 Python 进程内存，便于观察教学样本的资源占用。"""
    try:
        import psutil
        rss_gb = psutil.Process().memory_info().rss / (1024 ** 3)
        print(f"[内存] {stage}: {rss_gb:.2f} GiB")
    except Exception:
        pass


demo_handler_config = {
    "start_time": "2019-01-01",
    "end_time": "2019-01-31",
    "fit_start_time": "2019-01-01",
    "fit_end_time": "2019-01-31",
    "instruments": market,
}

print("探索区间：2019-01-01 → 2019-01-31")

**步骤 2：初始化数据处理器**

配置完成后，使用 `Alpha158` 初始化数据处理器。初始化时会自动：
- 加载原始数据
- 计算 158 个特征（价格、成交量、技术指标等）
- 生成标签（跨日收益率）
- 进行数据预处理（归一化、缺失值处理等）


In [ ]:
# ==================== 初始化一个月的演示处理器 ====================
from qlib.contrib.data.handler import Alpha158

# 这个小处理器只负责讲解因子与 raw / infer / learn，不参与正式训练。
demo_handler = Alpha158(**demo_handler_config)
handler = demo_handler
show_process_memory("一个月演示处理器初始化后")


#### Alpha158 数据处理器说明

`Alpha158` 是 Qlib 提供的预置数据处理器，它在初始化时会自动：

1. **构建 158 个特征**：通过 `Alpha158DL.get_feature_config()` 方法生成特征表达式配置 

2. **构建 1 个标签**：通过 `get_label_config()` 方法生成标签表达式
   - 默认标签表达式：`Ref($close, -2)/Ref($close, -1) - 1`

3. **数据预处理流程**：
   - 通过 `D.features` 计算表达式，生成原始数据（raw）
   - 在 `DataHandlerLP.process_data()` 方法中应用数据处理器：
     - **infer_processors=[]**：
       - 作用：处理特征数据，用于实际预测
       - Alpha158默认不处理特征（因为特征已在表达式层面相对化，如`KMID: ($close-$open)/$open`）
     - **learn_processors=[DropnaLabel, CSZScoreNorm]**：
       - 作用：处理标签数据，用于模型训练
       - `DropnaLabel`：删除标签为 NaN 的样本
       - `CSZScoreNorm`：对标签进行横截面标准化（只处理标签，不处理特征），即对每个交易日内的所有股票标签进行 Z-score 标准化，使每日标签均值为 0、标准差为 1。
   - 处理流程：`raw → shared_processors → infer_processors → _infer → learn_processors → _learn`
   - 生成三份数据：_data（原始数据 'raw'）、_infer（推理用处理后数据）、_learn（训练用处理后数据）



**理解 DataHandler 的三种数据模式**：

DataHandler 在初始化后会生成三种不同处理阶段的数据（raw、infer、learn），它们分别用于不同的场景。

**为什么需要区分这三种数据？**
- **训练时需要 learn 模式**：标签经过横截面标准化，便于模型学习相对强弱关系
- **预测时需要 infer 模式**：标签保持原始值，便于评估预测效果
- **查看原始数据 raw**：了解数据的原始分布和特征表达式

**为什么使用横截面标准化？**
1. **消除市场整体波动影响**：不同交易日市场整体涨跌不同，横截面标准化使每日标签分布一致
2. **便于模型训练**：标准化后的标签分布更稳定，有利于模型学习相对强弱关系
3. **保持相对排序**：标准化不改变股票间的相对排序，只调整分布

##### 特征(因子)查看

In [ ]:
# 从handler中获取因子配置
fields, names = handler.get_feature_config()

# 创建因子名称到表达式的映射
factor_dict = dict(zip(names, fields))

In [ ]:
# 查看特定因子
print(f"KMID: {factor_dict.get('KMID', '未找到')}")
print(f"MA5: {factor_dict.get('MA5', '未找到')}")

In [ ]:
# 查看所有因子的表达式
for name, expr in factor_dict.items():
    print(f"{name}: {expr}")

In [ ]:
# 获取一个月的特征样本，避免在教学展示阶段复制整套 Alpha158 数据
sample_period = slice("2019-01-01", "2019-01-31")
features = handler.fetch(selector=sample_period, col_set="feature")
print("特征样本形状:", features.shape)
features.head()

##### 标签查看

In [ ]:
# 查看 label 配置
print(handler.get_label_config())

In [ ]:
# 获取与特征相同时间段的标签样本
labels = handler.fetch(selector=sample_period, col_set="label")
print("标签样本形状:", labels.shape)
labels.head()

##### 数据预处理

In [ ]:
# ============================================
# 低内存方式：只比较一个月样本，不保留三份全量矩阵
# ============================================
import gc

sample_period = slice("2019-01-01", "2019-01-31")
mode_samples = {}

for data_key, label in (("raw", "原始"), ("infer", "推理"), ("learn", "学习")):
    sample = handler.fetch(selector=sample_period, data_key=data_key)
    mode_samples[data_key] = sample
    print(f"{label}数据样本形状: {sample.shape}")
    display(sample.head(2))

print("三种模式仅保留一个月样本；完整数据仍由 handler 统一管理。")

##### **课后作业（2）：理解三种数据模式（raw / infer / learn）**

下面的 `mode_samples` 来自独立的一个月演示处理器，足以比较三种模式；进入正式训练前会整体释放。

1. 比较三种样本的形状与缺失值数量；
2. 比较标签的均值和标准差；
3. 用文字说明 raw、infer、learn 分别适合什么场景。

In [ ]:
# 对比三种小样本的差异
print("数据形状对比:")
for key, frame in mode_samples.items():
    print(f"{key:>5}: {frame.shape}")

print("\n缺失值对比:")
# TODO: 请补充代码，分别计算三种样本的缺失值数量

print("\n标签统计对比:")
# TODO: 请补充代码，比较三种样本中 LABEL0 的均值与标准差

### 4.2 正式训练数据集放在第二部分

这里不创建覆盖多年数据的正式 `DatasetH`，避免讲解阶段的缓存进入训练阶段。请先完成上面的探索与练习，然后选择 **“运行时 → 断开连接并删除运行时”**，再打开《Qlib 模型训练、回测与绩效分析》。

## 下一步

第一部分到此结束。第二部分会在全新的 Colab 运行时中只创建一套 Alpha158，并完成模型训练、TopK 回测和绩效分析。